# 데이터 로드와 인덱싱

In [1]:
import pandas as pd, numpy as np, glob
from scipy.sparse import csr_matrix

def find(name):
    return (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(name))[0]
train = pd.read_csv(find('train.csv'))
test = pd.read_csv(find('test.csv'))
print(train.shape, train.userId.nunique(), train.movieId.nunique())

users = np.sort(train.userId.unique()); items = np.sort(train.movieId.unique())
u2i = {u: i for i, u in enumerate(users)}; m2i = {m: i for i, m in enumerate(items)}

def build(df):  # rating 무시하고 implicit(1)으로 변환
    X = csr_matrix((np.ones(len(df), dtype=np.float32),
                    (df.userId.map(u2i).values, df.movieId.map(m2i).values)),
                   shape=(len(users), len(items)))
    X.data[:] = 1
    return X

(80621, 4) 610 8237


# Validation Split

In [2]:
train = train.sort_values(['userId', 'time_idx'])
is_last = train.groupby('userId').cumcount(ascending=False) == 0
tr, va = train[~is_last], train[is_last]

# EASE

In [3]:
def ease(X, lam):
    G = (X.T @ X).toarray().astype(np.float64)
    G[np.diag_indices_from(G)] += lam
    P = np.linalg.inv(G)
    B = P / (-np.diag(P))
    B[np.diag_indices_from(B)] = 0
    return B

def recommend(X, B, uidx, k=20, bs=1000):
    out = []
    for s in range(0, len(uidx), bs):
        sub = X[uidx[s:s+bs]]
        S = np.asarray(sub @ B)
        S[sub.nonzero()] = -np.inf          # 이미 본 영화 제외
        top = np.argpartition(-S, k, axis=1)[:, :k]
        order = np.argsort(-np.take_along_axis(S, top, 1), axis=1)
        out.append(np.take_along_axis(top, order, 1))
    return np.vstack(out)

def ndcg_at_k(rec, uidx, truth, k=20):
    sc = []
    for row, u in zip(rec, uidx):
        t = truth[u]
        dcg = sum(1/np.log2(i+2) for i, it in enumerate(row[:k]) if it in t)
        idcg = sum(1/np.log2(i+2) for i in range(min(len(t), k)))
        sc.append(dcg / idcg)
    return np.mean(sc)

# 하이퍼파라미터 튜닝

In [4]:
Xtr = build(tr)
truth = va.assign(u=va.userId.map(u2i), m=va.movieId.map(m2i)).groupby('u')['m'].apply(set).to_dict()
vu = np.array(list(truth.keys()))
results = {}
for lam in [50, 100, 300, 500, 1000]:
    results[lam] = ndcg_at_k(recommend(Xtr, ease(Xtr, lam), vu), vu, truth)
    print(lam, results[lam])

50 0.0640534096037372
100 0.06467952156391532
300 0.06459563007882681
500 0.06203780363467533
1000 0.061602998838767985


# Train으로 다시 학습하기

In [5]:
best_lam = max(results, key=results.get)
print('best lam:', best_lam)
X = build(train)
B = ease(X, best_lam)
rec = recommend(X, B, test.userId.map(u2i).values)
sub = pd.DataFrame({'id': test.id,
                    'recommendations': [' '.join(map(str, items[r])) for r in rec]})
sub.to_csv('submission.csv', index=False)
sub.head()

best lam: 100


,id,recommendations
0,0,6990 8517 7554 3949 721 1010 5361 1008 5970 44...
1,1,1555 1065 7837 6596 2838 8173 6184 1111 2061 1...
2,2,2344 6990 4988 4480 5821 3949 2401 6298 586 85...
3,3,7332 6281 8819 290 8574 3713 6990 5219 1555 25...
4,4,6281 4115 1037 2480 5563 5842 5189 7714 3727 6...


# 모델과 추천함수

In [6]:
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, mf_dim=32, layers=(64, 32, 16)):
        super().__init__()
        self.mf_u = nn.Embedding(n_users, mf_dim); self.mf_i = nn.Embedding(n_items, mf_dim)
        self.mlp_u = nn.Embedding(n_users, layers[0] // 2); self.mlp_i = nn.Embedding(n_items, layers[0] // 2)
        mlp = []
        for a, b in zip(layers[:-1], layers[1:]):
            mlp += [nn.Linear(a, b), nn.ReLU()]
        self.mlp = nn.Sequential(*mlp)
        self.out = nn.Linear(mf_dim + layers[-1], 1)
        for e in [self.mf_u, self.mf_i, self.mlp_u, self.mlp_i]:
            nn.init.normal_(e.weight, std=0.01)

    def forward(self, u, i):
        g = self.mf_u(u) * self.mf_i(i)                                   # GMF
        m = self.mlp(torch.cat([self.mlp_u(u), self.mlp_i(i)], dim=-1))   # MLP
        return self.out(torch.cat([g, m], dim=-1)).squeeze(-1)            # NeuMF (logit)

@torch.no_grad()
def recommend_nn(model, X, uidx, k=20, bs=128):
    model.eval(); n_items = X.shape[1]; out = []
    all_i = torch.arange(n_items, device=dev)
    for s in range(0, len(uidx), bs):
        ub = uidx[s:s+bs]
        u = torch.tensor(ub, dtype=torch.long, device=dev).repeat_interleave(n_items)
        S = model(u, all_i.repeat(len(ub))).view(len(ub), n_items)
        r, c = X[ub].nonzero()
        S[torch.tensor(r, dtype=torch.long, device=dev), torch.tensor(c, dtype=torch.long, device=dev)] = -float('inf')  # 이미 본 영화 제외
        out.append(S.topk(k, dim=1).indices.cpu().numpy())
    model.train()
    return np.vstack(out)

device: cuda


# 학습 함수

In [7]:
def train_neumf(X, epochs, val=None, n_neg=4, bs=4096, lr=1e-3):
    n_users, n_items = X.shape
    model = NeuMF(n_users, n_items).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    pu, pi = X.nonzero()
    pos_keys = np.sort(pu.astype(np.int64) * n_items + pi)
    hist = []
    for ep in range(1, epochs + 1):
        # positive 1개당 negative n_neg개 샘플링 (본 영화가 뽑히면 다시 뽑기)
        nu = np.repeat(pu, n_neg); ni = np.random.randint(0, n_items, len(nu))
        for _ in range(5):
            key = nu.astype(np.int64) * n_items + ni
            idx = np.searchsorted(pos_keys, key).clip(max=len(pos_keys) - 1)
            bad = pos_keys[idx] == key
            if not bad.any(): break
            ni[bad] = np.random.randint(0, n_items, bad.sum())
        U = np.concatenate([pu, nu]); I = np.concatenate([pi, ni])
        Y = np.concatenate([np.ones(len(pu)), np.zeros(len(nu))]).astype(np.float32)
        perm = np.random.permutation(len(U))
        U = torch.tensor(U[perm], dtype=torch.long, device=dev)
        I = torch.tensor(I[perm], dtype=torch.long, device=dev)
        Y = torch.tensor(Y[perm], device=dev)
        tot = 0.0
        for s in range(0, len(U), bs):
            opt.zero_grad()
            loss = lossf(model(U[s:s+bs], I[s:s+bs]), Y[s:s+bs])
            loss.backward(); opt.step()
            tot += loss.item() * len(Y[s:s+bs])
        msg = f'epoch {ep}  loss {tot/len(U):.4f}'
        if val is not None:
            vu_, truth_ = val
            nd = ndcg_at_k(recommend_nn(model, X, vu_), vu_, truth_)
            hist.append(nd); msg += f'  val NDCG@20 {nd:.4f}'
        print(msg)
    return model, hist

# Validation으로 epoch 수 고르기

In [8]:
torch.manual_seed(0); np.random.seed(0)
model, hist = train_neumf(Xtr, 20, val=(vu, truth))
best_ep = int(np.argmax(hist)) + 1
print('best epoch:', best_ep, hist[best_ep - 1])

epoch 1  loss 0.5896  val NDCG@20 0.0283
epoch 2  loss 0.3614  val NDCG@20 0.0254
epoch 3  loss 0.3285  val NDCG@20 0.0262
epoch 4  loss 0.3204  val NDCG@20 0.0293
epoch 5  loss 0.3111  val NDCG@20 0.0300
epoch 6  loss 0.2982  val NDCG@20 0.0311
epoch 7  loss 0.2851  val NDCG@20 0.0292
epoch 8  loss 0.2716  val NDCG@20 0.0312
epoch 9  loss 0.2603  val NDCG@20 0.0315
epoch 10  loss 0.2474  val NDCG@20 0.0313
epoch 11  loss 0.2355  val NDCG@20 0.0323
epoch 12  loss 0.2229  val NDCG@20 0.0311
epoch 13  loss 0.2097  val NDCG@20 0.0333
epoch 14  loss 0.1979  val NDCG@20 0.0365
epoch 15  loss 0.1866  val NDCG@20 0.0348
epoch 16  loss 0.1760  val NDCG@20 0.0331
epoch 17  loss 0.1671  val NDCG@20 0.0326
epoch 18  loss 0.1584  val NDCG@20 0.0366
epoch 19  loss 0.1499  val NDCG@20 0.0364
epoch 20  loss 0.1427  val NDCG@20 0.0389
best epoch: 20 0.03890415932467598


# 전체 Train으로 다시 학습

In [9]:
torch.manual_seed(0); np.random.seed(0)
model_full, _ = train_neumf(X, best_ep)
rec = recommend_nn(model_full, X, test.userId.map(u2i).values)
sub = pd.DataFrame({'id': test.id,
                    'recommendations': [' '.join(map(str, items[r])) for r in rec]})
sub.to_csv('submission_neumf.csv', index=False)
sub.head()

epoch 1  loss 0.5885
epoch 2  loss 0.3597
epoch 3  loss 0.3282
epoch 4  loss 0.3195
epoch 5  loss 0.3079
epoch 6  loss 0.2931
epoch 7  loss 0.2794
epoch 8  loss 0.2679
epoch 9  loss 0.2569
epoch 10  loss 0.2451
epoch 11  loss 0.2340
epoch 12  loss 0.2216
epoch 13  loss 0.2096
epoch 14  loss 0.1977
epoch 15  loss 0.1871
epoch 16  loss 0.1772
epoch 17  loss 0.1671
epoch 18  loss 0.1587
epoch 19  loss 0.1512
epoch 20  loss 0.1444


,id,recommendations
0,0,6281 5361 3949 2344 4115 8819 9276 7263 3527 7...
1,1,1555 2401 1065 6596 7867 2061 8254 363 7304 32...
2,2,2344 6281 5361 4115 4885 8574 2401 7263 3949 7...
3,3,6281 7332 8819 846 5219 7899 7304 1948 8177 58...
4,4,6281 4115 9276 5361 5096 5842 8814 7714 84 372...
